# Presenter Solution: Reach Trajectory Speed Figures

This is a completed presenter copy of the worksheet. It is intentionally stored outside the tracked exercise folder so learners do not receive it as context.

By default, this notebook runs with a tiny demo dataset. To use real ReachX data, set `USE_DEMO_DATA = False` and set `SESSION_FOLDER` to a real session folder.


## 1. Imports And Constants


In [ ]:
from pathlib import Path
import re

from matplotlib.collections import LineCollection
from matplotlib import colors as mcolors
import matplotlib.pyplot as plt
import numpy as np

RESULT_NAMES = {
    2: "grabbed",
    3: "missed",
    4: "dropped",
    5: "stalled",
}

HAND_POS_NAMES = {
    0: "unspecified",
    1: "left",
    2: "right",
    3: "above",
    4: "below",
}

POINTS_PER_FRAME_PAIR = 8


## 2. User Settings


In [ ]:
notebook_folder = Path.cwd()
if notebook_folder.name == "presenter":
    output_folder = notebook_folder / "solution-figures"
else:
    output_folder = notebook_folder / "presenter" / "solution-figures"

USE_DEMO_DATA = True
SESSION_FOLDER = None
WORKSPACE_FORMAT = "auto"
PLOT_MODE = "single"
REACH_SELECTION = 1
PLOT_VIEW = "2d"
COLOR_MAP = "viridis"

print(f"Output folder: {output_folder}")
print(f"Demo mode: {USE_DEMO_DATA}")


## 3. Load Data

Demo mode creates a tiny in-memory dataset. Real-data mode loads a session folder, a curated reach file, and the matching trajectory output.


In [ ]:
if USE_DEMO_DATA:
    frame_numbers = np.arange(120)
    x_values = frame_numbers * 0.4
    y_values = 15.0 * np.sin(frame_numbers / 12.0)
    z_values = 8.0 * np.cos(frame_numbers / 14.0) + frame_numbers * 0.08
    xyz_values = np.column_stack([x_values, y_values, z_values])

    per_frame_distance = np.zeros(frame_numbers.shape[0])
    per_frame_distance[1:] = np.linalg.norm(np.diff(xyz_values, axis=0), axis=1)
    trajectory_xyz_speed = np.column_stack([xyz_values, per_frame_distance])

    curated_reach_rows = [
        {"frame": 12, "max_delta": 10, "duration": 24, "result": 2, "hand_pos": 2},
        {"frame": 50, "max_delta": 12, "duration": 28, "result": 3, "hand_pos": 2},
        {"frame": 86, "max_delta": 8, "duration": 22, "result": 2, "hand_pos": 2},
    ]
    speed_units = "mm/frame"
    session_label = "demo_session"

else:
    if SESSION_FOLDER is None or str(SESSION_FOLDER).strip() == "":
        raise ValueError("Set SESSION_FOLDER or use demo mode.")

    session_folder = Path(SESSION_FOLDER).expanduser()
    if not session_folder.is_dir():
        raise FileNotFoundError(f"Session folder does not exist: {session_folder}")

    reach_file_candidates = []
    for path in session_folder.glob("*_reaches.txt"):
        if "detected" in path.name.lower():
            continue
        if path.is_file():
            reach_file_candidates.append(path)

    if len(reach_file_candidates) != 1:
        raise ValueError(f"Expected one curated reach file, found {len(reach_file_candidates)}.")

    reach_file = reach_file_candidates[0]
    curated_reach_rows = []

    with reach_file.open("r", encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            values = [int(part) for part in stripped.split()]
            if len(values) == 4:
                values.append(0)
            if len(values) != 5:
                raise ValueError(f"Unexpected reach row: {stripped}")
            frame, max_delta, duration, result, hand_pos = values
            curated_reach_rows.append({
                "frame": frame,
                "max_delta": max_delta,
                "duration": duration,
                "result": result,
                "hand_pos": hand_pos,
            })

    fixed_scorer_folders = sorted(
        folder for folder in session_folder.iterdir()
        if folder.is_dir() and (folder / "trajectories.npz").is_file()
    )
    legacy_scorer_folders = sorted(
        folder for folder in session_folder.iterdir()
        if folder.is_dir() and (folder / "hand.npy").is_file()
    )

    detected_formats = []
    if fixed_scorer_folders:
        detected_formats.append("fixed")
    if legacy_scorer_folders:
        detected_formats.append("legacy")

    requested_format = str(WORKSPACE_FORMAT).strip().lower()
    if requested_format == "auto":
        if len(detected_formats) != 1:
            raise ValueError(f"Expected one detected workspace format, found {detected_formats}.")
        workspace_format = detected_formats[0]
    else:
        workspace_format = requested_format
        if workspace_format not in detected_formats:
            raise ValueError(f"Requested {workspace_format}, but detected {detected_formats}.")

    if workspace_format == "fixed":
        scorer_folder = fixed_scorer_folders[0]
        trajectory_file = scorer_folder / "trajectories.npz"
        with np.load(trajectory_file) as data:
            raw_trajectory = data["R_Hand"]
        trajectory_xyz_speed = raw_trajectory[:, [0, 1, 2, 6]]
        speed_units = "mm/ms"
    elif workspace_format == "legacy":
        scorer_folder = legacy_scorer_folders[0]
        trajectory_file = scorer_folder / "hand.npy"
        raw_trajectory = np.load(trajectory_file)
        trajectory_xyz_speed = raw_trajectory[:, [6, 1, 3, 10]]
        speed_units = "mm/s"
    else:
        raise ValueError(f"Unknown workspace format: {workspace_format}")

    session_label = reach_file.name.removesuffix("_reaches.txt")

print(f"Session: {session_label}")
print(f"Reach rows loaded: {len(curated_reach_rows)}")
print(f"Trajectory shape: {np.asarray(trajectory_xyz_speed).shape}")


## 4. Validate Curated Reach Rows


In [ ]:
required_reach_fields = ["frame", "max_delta", "duration", "result", "hand_pos"]

if not isinstance(curated_reach_rows, list):
    raise TypeError("curated_reach_rows should be a list of dictionaries.")
if len(curated_reach_rows) == 0:
    raise ValueError("No curated reaches were loaded.")

clean_reaches = []

for row_number, row in enumerate(curated_reach_rows, start=1):
    for field in required_reach_fields:
        if field not in row:
            raise ValueError(f"Reach row {row_number} is missing required field: {field}")

    frame = int(row["frame"])
    max_delta = int(row["max_delta"])
    duration = int(row["duration"])
    result = int(row["result"])
    hand_pos = int(row["hand_pos"])

    if frame < 0:
        raise ValueError(f"Reach row {row_number} has a negative frame number.")
    if max_delta < 0:
        raise ValueError(f"Reach row {row_number} has a negative max_delta.")
    if duration <= 0:
        raise ValueError(f"Reach row {row_number} has a non-positive duration.")
    if max_delta > duration:
        raise ValueError(f"Reach row {row_number} has max_delta greater than duration.")
    if result not in RESULT_NAMES:
        raise ValueError(f"Reach row {row_number} has an unknown result code: {result}")
    if hand_pos not in HAND_POS_NAMES:
        raise ValueError(f"Reach row {row_number} has an unknown hand_pos code: {hand_pos}")

    clean_reaches.append({
        "frame": frame,
        "max_delta": max_delta,
        "duration": duration,
        "result": result,
        "hand_pos": hand_pos,
        "source_row": row_number,
    })

print(f"Validated {len(clean_reaches)} curated reach rows.")


## 5. Sort Reaches And Check For Overlap


In [ ]:
clean_reaches = sorted(clean_reaches, key=lambda reach: reach["frame"])

previous_reach = None
for reach in clean_reaches:
    reach_end_frame = reach["frame"] + reach["duration"]
    if previous_reach is not None:
        previous_end_frame = previous_reach["frame"] + previous_reach["duration"]
        if reach["frame"] <= previous_end_frame:
            raise ValueError(
                "Curated reaches overlap: "
                f"row {previous_reach['source_row']} and row {reach['source_row']}."
            )
    previous_reach = reach

for reach_index, reach in enumerate(clean_reaches, start=1):
    reach["index"] = reach_index

for reach in clean_reaches:
    reach_end_frame = reach["frame"] + reach["duration"]
    print(
        f"{reach['index']:>3}. frame={reach['frame']} "
        f"end={reach_end_frame} "
        f"result={RESULT_NAMES[reach['result']]} "
        f"hand_pos={HAND_POS_NAMES[reach['hand_pos']]}"
    )


## 6. Validate The Trajectory Array


In [ ]:
trajectory_xyz_speed = np.asarray(trajectory_xyz_speed)

if trajectory_xyz_speed.ndim != 2:
    raise ValueError("trajectory_xyz_speed should be a 2D NumPy array.")
if trajectory_xyz_speed.shape[1] != 4:
    raise ValueError(f"trajectory_xyz_speed should have four columns. Found {trajectory_xyz_speed.shape}.")
if trajectory_xyz_speed.shape[0] < 2:
    raise ValueError("trajectory_xyz_speed has fewer than two frames.")
if not np.isfinite(trajectory_xyz_speed).all():
    raise ValueError("trajectory_xyz_speed contains non-finite values.")

print(f"Trajectory frames: {trajectory_xyz_speed.shape[0]}")
print(f"Speed units: {speed_units}")


## 7. Summarize Loaded Data


In [ ]:
loaded_data_summary = [
    f"Session: {session_label}",
    f"Reach rows: {len(clean_reaches)}",
    f"Trajectory frames: {trajectory_xyz_speed.shape[0]}",
]

speed_min = float(np.min(trajectory_xyz_speed[:, 3]))
speed_max = float(np.max(trajectory_xyz_speed[:, 3]))
loaded_data_summary.append(f"Speed range: {speed_min:.3f} to {speed_max:.3f} {speed_units}")

for line in loaded_data_summary:
    print(line)


## 8. Add Reach Details


In [ ]:
reach_detail_rows = []

for reach in clean_reaches:
    peak_frame = reach["frame"] + reach["max_delta"]
    end_frame = reach["frame"] + reach["duration"]
    reach_detail_rows.append({
        "index": reach["index"],
        "frame": reach["frame"],
        "peak_frame": peak_frame,
        "end_frame": end_frame,
        "result": RESULT_NAMES[reach["result"]],
    })

for row in reach_detail_rows[:5]:
    print(row)


## 9. Select Reaches To Plot


In [ ]:
plot_mode = str(PLOT_MODE).strip().lower()

if plot_mode == "all":
    selected_reaches = clean_reaches
elif plot_mode == "single":
    selected_value = int(REACH_SELECTION)

    if 1 <= selected_value <= len(clean_reaches):
        selected_reaches = [clean_reaches[selected_value - 1]]
    else:
        matching_reaches = [reach for reach in clean_reaches if reach["frame"] == selected_value]
        if len(matching_reaches) != 1:
            raise ValueError(f"No single reach matched selection: {selected_value}")
        selected_reaches = matching_reaches
else:
    raise ValueError("PLOT_MODE should be 'single' or 'all'.")

print(f"Selected reach indices: {[reach['index'] for reach in selected_reaches]}")


## 10. Extract Reach Segments


In [ ]:
reach_segments = []

for reach in selected_reaches:
    start_frame = reach["frame"]
    end_frame = reach["frame"] + reach["duration"]

    if end_frame >= trajectory_xyz_speed.shape[0]:
        raise ValueError(
            f"Reach {reach['index']} ends at frame {end_frame}, "
            f"but trajectory has only {trajectory_xyz_speed.shape[0]} frames."
        )

    segment = trajectory_xyz_speed[start_frame:end_frame + 1, :]
    reach_segments.append({"reach": reach, "segment": segment})

print(f"Prepared {len(reach_segments)} reach segment(s).")


## 11. Calculate A Simple Reach Metric


In [ ]:
reach_metrics = []

for item in reach_segments:
    reach = item["reach"]
    segment = item["segment"]

    position_steps = np.diff(segment[:, :3], axis=0)
    path_length_mm = float(np.sum(np.linalg.norm(position_steps, axis=1)))
    max_speed = float(np.max(segment[:, 3]))

    reach_metrics.append({
        "index": reach["index"],
        "path_length_mm": path_length_mm,
        "max_speed": max_speed,
    })

reach_metrics


## 12. Plot Reach Paths


In [ ]:
plot_view = str(PLOT_VIEW).strip().lower()

all_segment_speeds = np.concatenate([item["segment"][:, 3] for item in reach_segments])
speed_min = float(np.min(all_segment_speeds))
speed_max = float(np.max(all_segment_speeds))
if speed_min == speed_max:
    speed_max = speed_min + 1.0

if plot_mode == "all":
    title_reach_text = f"all curated reaches (n={len(reach_segments)})"
else:
    selected = reach_segments[0]["reach"]
    title_reach_text = f"reach {selected['index']} frame {selected['frame']}"

figure_title = f"{session_label} | {plot_view.upper()} | {title_reach_text}"

if plot_view == "2d":
    speed_norm = mcolors.Normalize(vmin=speed_min, vmax=speed_max)
    fig, axis = plt.subplots(figsize=(7, 6))
    color_source = None

    for reach_number, item in enumerate(reach_segments):
        segment = item["segment"]
        y_values = segment[:, 1]
        z_values = segment[:, 2]
        speed_values = segment[:, 3]

        points = np.column_stack([y_values, z_values])
        smooth_points = []
        smooth_speeds = []

        for index in range(points.shape[0] - 1):
            for step in range(POINTS_PER_FRAME_PAIR):
                fraction = step / POINTS_PER_FRAME_PAIR
                smooth_points.append(points[index] + fraction * (points[index + 1] - points[index]))
                smooth_speeds.append(speed_values[index] + fraction * (speed_values[index + 1] - speed_values[index]))

        smooth_points.append(points[-1])
        smooth_speeds.append(speed_values[-1])
        smooth_points = np.asarray(smooth_points)
        smooth_speeds = np.asarray(smooth_speeds)

        segments = np.stack([smooth_points[:-1], smooth_points[1:]], axis=1)
        segment_speeds = (smooth_speeds[:-1] + smooth_speeds[1:]) / 2.0

        collection = LineCollection(segments, cmap=COLOR_MAP, norm=speed_norm, linewidth=2.0)
        collection.set_array(segment_speeds)
        axis.add_collection(collection)
        color_source = collection

        start_label = "start" if reach_number == 0 else None
        end_label = "end" if reach_number == 0 else None
        axis.scatter(y_values[0], z_values[0], color="black", s=24, marker="o", label=start_label, zorder=3)
        axis.scatter(y_values[-1], z_values[-1], color="black", s=24, marker="s", label=end_label, zorder=3)

    axis.autoscale()
    axis.set_aspect("equal", adjustable="datalim")
    axis.set_xlabel("Y position (mm)")
    axis.set_ylabel("Z position (mm)")
    axis.set_title(figure_title)
    axis.legend(loc="best")

    colorbar = fig.colorbar(color_source, ax=axis, shrink=0.8)
    colorbar.set_label(f"Speed ({speed_units})")
    fig.tight_layout()

elif plot_view == "3d":
    import plotly.graph_objects as go

    fig = go.Figure()

    for reach_number, item in enumerate(reach_segments):
        reach = item["reach"]
        segment = item["segment"]
        points = segment[:, :3]
        speed_values = segment[:, 3]

        smooth_points = []
        smooth_speeds = []
        for index in range(points.shape[0] - 1):
            for step in range(POINTS_PER_FRAME_PAIR):
                fraction = step / POINTS_PER_FRAME_PAIR
                smooth_points.append(points[index] + fraction * (points[index + 1] - points[index]))
                smooth_speeds.append(speed_values[index] + fraction * (speed_values[index + 1] - speed_values[index]))
        smooth_points.append(points[-1])
        smooth_speeds.append(speed_values[-1])
        smooth_points = np.asarray(smooth_points)
        smooth_speeds = np.asarray(smooth_speeds)

        trace_name = f"reach {reach['index']} frame {reach['frame']}"
        line_settings = {
            "color": smooth_speeds,
            "colorscale": "Viridis",
            "cmin": speed_min,
            "cmax": speed_max,
            "width": 6,
            "showscale": reach_number == 0,
        }
        if reach_number == 0:
            line_settings["colorbar"] = {"title": f"Speed ({speed_units})"}

        fig.add_trace(
            go.Scatter3d(
                x=smooth_points[:, 0],
                y=smooth_points[:, 1],
                z=smooth_points[:, 2],
                mode="lines",
                name=trace_name,
                line=line_settings,
                customdata=smooth_speeds,
                hovertemplate=(
                    f"{trace_name}<br>"
                    "X: %{x:.3f} mm<br>"
                    "Y: %{y:.3f} mm<br>"
                    "Z: %{z:.3f} mm<br>"
                    "Speed: %{customdata:.5f}<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title=figure_title,
        scene={
            "xaxis_title": "X position (mm)",
            "yaxis_title": "Y position (mm)",
            "zaxis_title": "Z position (mm)",
            "aspectmode": "data",
        },
        margin={"l": 0, "r": 0, "t": 50, "b": 0},
    )

else:
    raise ValueError("PLOT_VIEW should be '2d' or '3d'.")

fig


## 13. Save The Figure


In [ ]:
output_folder.mkdir(exist_ok=True)

safe_session_label = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(session_label)).strip("_")
if not safe_session_label:
    safe_session_label = "worksheet_session"

if plot_mode == "all":
    output_stem = f"{safe_session_label}_{plot_view}_all_reaches"
else:
    reach = selected_reaches[0]
    output_stem = f"{safe_session_label}_{plot_view}_reach{reach['index']:03d}_frame{reach['frame']}"

saved_files = []

if plot_view == "2d":
    png_file = output_folder / f"{output_stem}.png"
    pdf_file = output_folder / f"{output_stem}.pdf"
    fig.savefig(png_file, dpi=300)
    fig.savefig(pdf_file)
    saved_files = [png_file, pdf_file]
else:
    html_file = output_folder / f"{output_stem}.html"
    fig.write_html(str(html_file), include_plotlyjs="cdn", full_html=True)
    saved_files = [html_file]

for saved_file in saved_files:
    print(saved_file)
